In [0]:
from pyspark.sql import functions as F

dbutils.widgets.text("load_date", "2026-09-23")
LOAD_DATE = dbutils.widgets.get("load_date")
CATALOG = "olist"
LANDING = f"/Volumes/{CATALOG}/raw/landing/load_date={LOAD_DATE}"
for f in dbutils.fs.ls(LANDING):
    if not f.name.endswith(".csv"):
        continue
    table = f.name.replace("olist_", "").replace("_dataset.csv", "").replace(".csv", "")

    df = (spark.read
          .option("header", True)
          .option("multiLine", True)   # review comments span lines
          .option("escape", '"')
          .csv(f.path)                 # keep everything as strings in bronze
          .withColumn("_source_file", F.col("_metadata.file_path"))
          .withColumn("_ingested_at", F.current_timestamp())
          .withColumn("_load_date", F.lit(LOAD_DATE)))
    
    (df.write.mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.bronze.{table}"))

    print(f"{table}: {df.count():,} rows")